In [ ]:
import warnings
warnings.simplefilter('ignore')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import auc, roc_curve, classification_report, confusion_matrix
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold

import h2o
from h2o.frame import H2OFrame
from h2o.estimators.random_forest import H2ORandomForestEstimator
from h2o.grid.grid_search import H2OGridSearch
from h2o.estimators.gbm import H2OGradientBoostingEstimator

## 导入数据

In [ ]:
user = pd.read_csv('user.csv')
receipt = pd.read_csv('receipt_table.csv')
returning = pd.read_csv('return_table.csv')
user.head()

In [ ]:
user.info()

In [ ]:
receipt.describe()

In [ ]:
receipt.info()

In [ ]:
returning.describe()

In [ ]:
returning.info()

## 数据预处理

In [ ]:
#首先借据数量和用户数量并不一致，排查原因
print("Unique user in receipt table:", receipt['user_id'].nunique())
print("Unique user in user info table:", user['user_id'].nunique())

In [ ]:
#借据中独特用户比用户信息表中还少，那就意味着有一对多
receipt['user_id'].value_counts().head()

In [ ]:
#确认有一对多，但我们要做的模型是一个违约风险预测，所以我们其实测量标准是每一单贷款的还款，后续回头检查这些个例对总体的影响
user_data = pd.merge(receipt, user, how = 'left', on = 'user_id')
user_data.head()

In [ ]:
user_data.info()

In [ ]:
#最后处理流水信息，考虑哪些要加入那些去除，加入的又要怎么操作
#首先应用我们的窗口期2022/1/1 - 2024/12/31三年观察期预测两个月后2025/3/1时是否有逾期
returning['release_date'] = pd.to_datetime(returning['release_date'])
returning['lease_due'] = pd.to_datetime(returning['lease_due'])

obs_start = pd.to_datetime('2022-01-01')
obs_end = pd.to_datetime('2025-01-01')

cond1 = (returning['release_date'] >= obs_start) & (returning['release_date'] < obs_end)

cond2 = (returning['release_date'] < obs_start) & (returning['lease_due'] >= obs_start)

cond3 = (returning['release_date'] < obs_start) & (returning['lease_due'].isna())

returning_filtered = returning[cond1 | cond2 | cond3]

returning_filtered['Date'] = pd.to_datetime(returning_filtered['Date'])
returning_filtered = returning_filtered[(returning_filtered['Date'] >= obs_start) & (returning_filtered['Date'] < obs_end)]

returning_filtered.head()

In [ ]:
returning_filtered.info()

In [ ]:
#所有的借据和流水都在窗口期内
#检查一下，有多少个独特的用户在流水中
print("Unique receipt in receipt table:", user_data['user_id'].nunique())
print("Unique receipt in returning table:", returning_filtered['user_id'].nunique())

In [ ]:
#接下来聚合流水信息形成静态数据
returning_filtered['days_to_obs_end'] = (obs_end - returning_filtered['Date']).dt.days
user_repay_features = returning_filtered.groupby('user_id').agg(
    repay_total=('returned', 'sum'), #总还款,后续用来算repay ratio
    repay_count=('returned', 'count'), #还款次数
    repay_std=('returned', 'std'), #还款波动值
    repay_max=('returned', 'max'), #最高单次还款
    
    repay_span=('Date', lambda x: (x.max() - x.min()).days if len(x) > 1 else 0) #最后一次还款与第一次还款跨度
    #### 这份数据里转本金全部为0，我就暂且不做这部分了
    #capitalized_count=('Capitalized', 'sum'), #转本金次数
    #capitalized_ratio=('Capitalized', 'mean') #转本金率
).reset_index()
user_repay_features.head()

In [ ]:
#与之前的静态数据结合形成需要的表格
df = user_data.merge(user_repay_features, on='user_id', how='left')
df.head()

In [ ]:
#还款特征缺失值填充
repay_cols = [col for col in df.columns if col.startswith('repay_') 
              or col.startswith('capitalized_') 
              or col.startswith('days_since') 
              or col.startswith('return_type')]

df[repay_cols] = df[repay_cols].fillna(0)
df.info()

In [ ]:
#删除无用列
df = df.drop(['family_member', 'amount_due', 'receipt_id', 'province', 'city'], axis = 1)
df.info()

## DLC:加入另外时间段测试集

In [ ]:
user_test = pd.read_csv('user_test.csv')
receipt_test = pd.read_csv('receipt_test.csv')
return_test = pd.read_csv('return_test.csv')
user_test.head()

In [ ]:
user_test.info()

In [ ]:
return_test.info()

In [ ]:
user_data_test = pd.merge(receipt_test, user_test, how = 'left', on = 'user_id')
user_data_test.head()

In [ ]:
user_data_test['is_test'] = 1
user_data_test.head()

In [ ]:
#最后处理流水信息，考虑哪些要加入那些去除，加入的又要怎么操作
return_test['release_date'] = pd.to_datetime(return_test['release_date'])
return_test['lease_due'] = pd.to_datetime(return_test['lease_due'])

obs_start = pd.to_datetime('2024-04-01')
obs_end = pd.to_datetime('2025-03-31')

cond1 = (return_test['release_date'] >= obs_start) & (return_test['release_date'] < obs_end)

cond2 = (return_test['release_date'] < obs_start) & (return_test['lease_due'] >= obs_start)

cond3 = (return_test['release_date'] < obs_start) & (return_test['lease_due'].isna())

return_test_filtered = return_test[cond1 | cond2 | cond3]

return_test_filtered['Date'] = pd.to_datetime(return_test_filtered['Date'])
return_test_filtered = return_test_filtered[(return_test_filtered['Date'] >= obs_start) & (return_test_filtered['Date'] < obs_end)]

return_test_filtered.head()

In [ ]:
return_test_filtered.info()

In [ ]:
#接下来聚合流水信息形成静态数据
return_test_filtered['days_to_obs_end'] = (obs_end - return_test_filtered['Date']).dt.days
user_repay_test_features = return_test_filtered.groupby('user_id').agg(
    repay_total=('returned', 'sum'), #总还款,后续用来算repay ratio
    repay_count=('returned', 'count'), #还款次数
    repay_std=('returned', 'std'), #还款波动值
    repay_max=('returned', 'max'), #最高单次还款
    
    repay_span=('Date', lambda x: (x.max() - x.min()).days if len(x) > 1 else 0) #最后一次还款与第一次还款跨度
    #### 这份数据里转本金全部为0，我就暂且不做这部分了
    #capitalized_count=('Capitalized', 'sum'), #转本金次数
    #capitalized_ratio=('Capitalized', 'mean') #转本金率
).reset_index()
user_repay_test_features.head()

In [ ]:
#与之前的静态数据结合形成需要的表格
df = pd.concat([df, user_data_test.merge(user_repay_test_features, on='user_id', how='left')], ignore_index = True)
df.head()

In [ ]:
df.info()

In [ ]:
#删除无用列
df = df.drop(['family_member', 'amount_due', 'receipt_id', 'province', 'city'], axis = 1)
df.info()

## 年龄

In [ ]:
#处理年龄特征
df['birth'] = pd.to_datetime(user_data['birth'], errors='coerce')
#df['birth'].isna().sum()
df['age'] = (obs_end - df['birth']).dt.days // 365
invalid_birth_rows = df[df['birth'].isna()]
#print(df['age'])
#df.drop('birth', axis = 1, inplace = True)
#df.head()
invalid_birth_rows

In [ ]:
#用平均年龄处理缺失值
mean_age = df['age'].mean()
df['age'].fillna(round(mean_age), inplace = True)
df = df.drop('birth', axis = 1)
df.head()

In [ ]:
#业务分箱
bins = [18, 30, 45, 60, 100]
df['age_bin'] = pd.cut(df['age'], bins=bins, labels=['teenager', 'youngs','mid_age','old'])
#引入二次项来强化模型非线性识别
df['age_squared'] = (df['age'] - 30)**2
#对分箱生成独热编码并将要使用的特征存入单独的dataframe
#features = pd.get_dummies(df['age_bin'], prefix='age')
features = pd.DataFrame(df[['age_squared', 'age', 'age_bin']])
#features = features.join(df[['age_squared', 'age_scaled', 'age_bin']])
#为使用梯度提升树模型，转化为category类型
features['age_bin'] = features['age_bin'].astype('category')
features.head()

In [ ]:
df.head()

## 流水

In [ ]:
df['repay_ratio'] = df['repay_total'] / df['amount']
features = features.join(df[['repay_ratio', 'repay_count', 'repay_std', 'repay_max', 'repay_span']])
features.head()

## 婚姻状况

In [ ]:
#创建mapping
marital_mapping = {
    21: "first_time_marrage",
    22: "marriged",
    23: "marriged",
    30: "windowed",
    31: "divorced",
    90: "missing",
    10: "single",
    20: "marriged"
}
df['marital_status'] = df['marital_status'].map(marital_mapping)
#使用方法中想要合并某些稀有类型来降本增效
df['marital_status'] = df['marital_status'].replace(
        ['divorced', 'windowed'], 
        'divorced/widowed'
    )
#为使用树模型，将缺失值统一处理为单独类型
df['marital_status'] = df['marital_status'].fillna('missing')
features = features.join(df['marital_status'])
features['marital_status'] = features['marital_status'].astype('category')
df.head()

In [ ]:
features.head()

## 最高学历

In [ ]:
#为体现城市影响，先引入城市等级
CITY_CLASS_MAPPING = {
    # 一线城市
    '北京': 'tier1', '上海': 'tier1', '广州': 'tier1', '深圳': 'tier1',
    '成都': 'tier1', '杭州': 'tier1', '重庆': 'tier1', '武汉': 'tier1',
    '西安': 'tier1', '苏州': 'tier1', '天津': 'tier1', '南京': 'tier1',
    '长沙': 'tier1', '郑州': 'tier1', '青岛': 'tier1', '合肥': 'tier1',
    '东莞': 'tier1', '宁波': 'tier1', '佛山': 'tier1', 
    #二线城市
    '济南': 'tier2', '昆明': 'tier2', '无锡': 'tier2', '沈阳': 'tier2',
    '福州': 'tier2', '厦门': 'tier2', '温州': 'tier2', '石家庄': 'tier2',
    '大连': 'tier2', '金华': 'tier2', '泉州': 'tier2', '哈尔滨': 'tier2',
    '南宁': 'tier2', '长春': 'tier2', '南昌': 'tier2', '常州': 'tier2',
    '南通': 'tier2', '贵阳': 'tier2', '嘉兴': 'tier2', '徐州': 'tier2',
    '惠州': 'tier2', '太原': 'tier2', '烟台': 'tier2', '临沂': 'tier2',
    '保定': 'tier2', '台州': 'tier2', '绍兴': 'tier2', '珠海': 'tier2',
    '洛阳': 'tier2', '潍坊': 'tier2',
    #三线城市
    '乌鲁木齐': 'tier3', '兰州': 'tier3', '中山': 'tier3', '盐城': 'tier3', 
    '海口': 'tier3', '扬州': 'tier3', '济宁': 'tier3', '湖州': 'tier3', 
    '赣州': 'tier3', '邯郸': 'tier3', '南阳': 'tier3', '唐山': 'tier3', 
    '芜湖': 'tier3', '阜阳': 'tier3', '廊坊': 'tier3', '汕头': 'tier3', 
    '泰州': 'tier3', '呼和浩特': 'tier3', '镇江': 'tier3', '江门': 'tier3', 
    '菏泽': 'tier3', '连云港': 'tier3', '沧州': 'tier3', '淄博': 'tier3', 
    '新乡': 'tier3', '同口': 'tier3', '襄阳': 'tier3', '淮安': 'tier3', 
    '商丘': 'tier3', '桂林': 'tier3', '咸阳': 'tier3', '上饶': 'tier3', 
    '银川': 'tier3', '宿迁': 'tier3', '漳州': 'tier3', '遵义': 'tier3', 
    '荆州': 'tier3', '绵阳': 'tier3', '宜昌': 'tier3', '威海': 'tier3', 
    '湛江': 'tier3', '九江': 'tier3', '邢台': 'tier3', '揭阳': 'tier3', 
    '三亚': 'tier3', '衡阳': 'tier3', '信阳': 'tier3', '泰安': 'tier3', 
    '肇庆': 'tier3', '蚌埠': 'tier3', '安阳': 'tier3', '安庆': 'tier3', 
    '德州': 'tier3', '株洲': 'tier3', '莆田': 'tier3', '聊城': 'tier3', 
    '驻马店': 'tier3', '岳阳': 'tier3', '亳州': 'tier3', '鄞州': 'tier3', 
    '宜春': 'tier3', '宿州': 'tier3', '黄冈': 'tier3', '六安': 'tier3', 
    '常德': 'tier3', '宁德': 'tier3', '茂名': 'tier3', '马鞍山': 'tier3', 
    '衢州': 'tier3', '枣庄': 'tier3', '宜宾': 'tier3', '榆林': 'tier3', 
    '开封': 'tier3', '邵阳': 'tier3', '运城': 'tier3', '清远': 'tier3', 
    '吉安': 'tier3', '日照': 'tier3', '许昌': 'tier3', '包头': 'tier3', 
    '郴州': 'tier3', '滨州': 'tier3', '丽水': 'tier3', '宣城': 'tier3', 
    '淮南': 'tier3', '平顶山': 'tier3', '东营': 'tier3', '南充': 'tier3', 
    '秦皇岛': 'tier3', '黄石': 'tier3', '鞍山': 'tier3', '晋中': 'tier3', 
    '曲靖': 'tier3', '孝感': 'tier3', '抚州': 'tier3', '宝鸡': 'tier3', 
    '渭南': 'tier3', '舟山': 'tier3', '德阳': 'tier3', '衡水': 'tier3', 
    '吉林': 'tier3', '西宁': 'tier3', '龙岩': 'tier3', '焦作': 'tier3', 
    '十堰': 'tier3', '濮阳': 'tier3', '潮州': 'tier3', '大庆': 'tier3', 
    '渑潭': 'tier3', '鄂尔多斯': 'tier3', '泸州': 'tier3', '长治': 'tier3', 
    '怀化': 'tier3', '玉林': 'tier3', '大同': 'tier3', '梅州': 'tier3', 
    '南平': 'tier3', '黄山': 'tier3', '临汾': 'tier3', '赤峰': 'tier3', 
    '恩施': 'tier3', '齐齐哈尔': 'tier3', '张家口': 'tier3', '阳江': 'tier3', 
    '达州': 'tier3', '乐山': 'tier3', '益阳': 'tier3', '汕尾': 'tier3', 
    '大理': 'tier3', '永州': 'tier3', '红河': 'tier3', '北海': 'tier3', 
    '河源': 'tier3', '锦州': 'tier3', '毕节': 'tier3', '梁德镇': 'tier3', 
    '晋城': 'tier3', '凉山': 'tier3', '韶关': 'tier3', '三明': 'tier3', 
    '黔东南': 'tier3', '眉山': 'tier3', '承德': 'tier3', '铜陵': 'tier3', 
    '荆门': 'tier3', '黔南': 'tier3', '淮北': 'tier3', '铜仁': 'tier3', 
    '咸宁': 'tier3', '曾口': 'tier3', '娄底': 'tier3', '汉中': 'tier3', 
    '玉溪': 'tier3', '喀什': 'tier3', '遂宁': 'tier3', '拉萨': 'tier3', 
    '百色': 'tier3', '天水': 'tier3', '吕梁': 'tier3', '巫峡': 'tier3',
    '滁州': 'tier3'
}
#现在给数据集里面的地址打城市标签
city_tier = []
df['address'] = df['address'].astype(str)
for addr in list(df['address']):
    #print(addr)
    flag = False
    for city, tier in CITY_CLASS_MAPPING.items():
        if city in addr:
            city_tier.append(tier)
            flag = True
            break
    if not flag:
        city_tier.append('rural')
df['city_tier'] = city_tier
df.drop('address', axis = 1, inplace = True)
df.head()

In [ ]:
edu_mapping = {
    90 : 'illiterate',
    99 : 'missing',
    40 : 'secondary_specialized_school',
    80 : 'primary_school',
    70 : 'junior_high_school',
    60 : 'senior_high_school',
    20 : 'bachelor',
    10 : 'master',
    30 : 'junior_college',
    50 : 'specialized_school'
}
edu_experience = []
for i in df['edu']:
    if i in edu_mapping:
        edu_experience.append(edu_mapping[i])
    else:
        edu_experience.append('missing')
df['edu'] = edu_experience
df['edu_city_interaction'] = (
            df['edu'].astype(str) + 
            '_in_' + 
            df['city_tier'].astype(str)
        )
df.head()

In [ ]:
features = features.join(df[['edu', 'edu_city_interaction']])
features['edu'] = features['edu'].astype('category')
features['edu_city_interaction'] = features['edu_city_interaction'].astype('category')
features.head()

## 贷款用途

In [ ]:
usage_mapping = {
    53: "BorrowNewRepayOld",
    1: "BuyNewCommercialHousing",
    3: "BuySecondhandCommercialHousing",
    2: "BuyAffordableHousing",
    13: "BuyPriceLimitedHousing",
    4: "BuySecondaryPlusHousing",
    6: "BuyVilla",
    9: "BuyNewRetailProperty",
    10: "BuySecondhandRetailProperty",
    11: "BuyNewOfficeProperty",
    12: "BuySecondhandOfficeProperty",
    8: "BuyResidentialCommercialCombo",
    16: "BuySedanCar",
    19: "BuySUV",
    15: "BuyMinibus",
    20: "BuyImportedVehicle",
    26: "BuyOtherConsumerVehicles",
    17: "BuyLargeMediumBus",
    18: "BuyTruck",
    21: "BuyPassengerCargoCombo",
    24: "BuyTaxi",
    25: "BuyConstructionVehicle",
    22: "BuyOtherCommercialVehicles",
    27: "BuyConstructionMachinery",
    37: "StudyAbroadExpenses",
    38: "StudyAbroadDeposit",
    36: "NonCompulsoryEducationFees",
    35: "CompulsoryEducationFees",
    41: "ResidentialRenovation",
    42: "CommercialRenovationNonBusiness",
    51: "PurchaseRawMaterials",
    57: "CommodityProcurement",
    52: "PurchaseEquipment",
    54: "PayProjectFees",
    58: "TechnicalUpgrading",
    32: "OtherPersonalBusinessUse",
    60110: "FisheryExpenditure",
    60120: "ForestryExpenditure",
    60130: "SidelineExpenditure",
    60210: "BreedingExpenditure",
    60310: "PlantingExpenditure",
    43: "WeddingCeremony",
    44: "TravelExpenses",
    40: "BuyDurableConsumerGoods",
    48: "OtherInvestments",
    49: "OtherPersonalConsumption",
    56: "CapitalConstruction",
    61: "ConstructOfficeBuilding",
    62: "ConstructFactoryBuilding",
}
usage_cate = []
for i in df['usage']:
    if i in usage_mapping:
        usage_cate.append(usage_mapping[i])
    else:
        usage_cate.append('unknown')
df['usage'] = usage_cate
df['usage_city_interaction'] = (
            df['usage'].astype(str) + 
            '_in_' + 
            df['city_tier'].astype(str)
)
df.head()

In [ ]:
features = features.join(df[['usage', 'usage_city_interaction']])
features['usage'] = features['usage'].astype('category')
features['usage_city_interaction'] = features['usage_city_interaction'].astype('category')
features.head()

## 职业

In [ ]:
#定义职业码值表
job_mapping = {
    'X' : 'Military',
    'Y' : 'Other',
    'Z' : 'Unknown',
    '0' : 'Government',
    '1' : 'Tech',
    '3' : 'Office_staff',
    '4' : 'Commercial&Service',
    '5' : 'Production',
    '6' : 'Operator'
}
#职称码值
pos_mapping = {
    2 : 'junior',
    1 : 'senior',
    9 : 'missing',
    0 : 'none',
    3 : 'freshman'
}
#将码值打入表格替换原码值并引入地理位置特征
job_cate = []
for i in df['job']:
    if i in job_mapping:
        job_cate.append(job_mapping[i])
    else:
        job_cate.append('Missing')
df['job'] = job_cate
df['job_city_interaction'] = (
            df['job'].astype(str) + 
            '_in_' + 
            df['city_tier'].astype(str)
)
#打入表格并引入职业-职称
df['pos'] = df['pos'].map(pos_mapping)
df['job_pos'] = (
            df['pos'].astype(str) + 
            '_in_' + 
            df['job'].astype(str)
)
df.head()

In [ ]:
features = features.join(df[['job', 'job_city_interaction', 'job_pos']])
features['job'] = features['job'].astype('category')
features['job_city_interaction'] = features['job_city_interaction'].astype('category')
features['job_pos'] = features['job_pos'].astype('category')
features.head()

## 收入

In [ ]:
#其他的收入工程有些问题待确认，先做家庭平均收入
#首先缺失值可能代表信息，加入两个特征为是否空缺
df['income_missing'] = df['family_income'].isna().astype(int)
#接下来填充缺失值，使用数据根据地区教育职业分组后的中位数
group_cols = ['marital_status', 'city_tier']  # 选择与收入相关性强的特征
#目前会出现很多NaN因为数据量不够，如果是大量数据大概率会找到median
df['family_income_filled'] = df.groupby(group_cols)['family_income'].transform(lambda x: x.fillna(x.median()))
df.loc[df['family_income_filled'] == 0, 'family_income_filled'] = 1
df['log_family_income'] = np.log(df['family_income_filled'])

In [ ]:
#标记收入缺失，收入缺失可能以为低收入或极高收入或高风险收入
df['income_missing'] = df['income'].isna().astype(int)
#将个人收入取log放入特征，保证不会有过高的收入噪音对模型造成大影响
group_cols = ['job_pos', 'edu_city_interaction']
#同样使用工作教育和地区的组合特征分组中位数（收入一般是偏态分布）来填缺失值
df['income_filled'] = df.groupby(group_cols)['income'].transform(lambda x: x.fillna(x.median()))
df['income_filled'] = df.groupby('job_pos')['income'].transform(lambda x: x.fillna(x.median()))
df['income_filled'] = df.groupby('edu_city_interaction')['income'].transform(lambda x: x.fillna(x.median()))
df.loc[df['income_filled'] == 0, 'income_filled'] = 1
df['log_income'] = np.log(df['income_filled'])
df.head()

In [ ]:
features = features.join(df[['log_income', 'income_missing', 'log_family_income']])
features.head()

In [ ]:
features['log_family_income'] = features['log_family_income'].fillna(1)
features.info()

## 借贷金额

In [ ]:
#首先引入log金额来减小极大金额贷款对模型的影响
df['log_loan_amount'] = np.log(df['amount'])
#最后引入地区购买力特征，即在三线城市借贷1w相当于在一线城市借了1w3的购买力
region_mapping = {
    'tier1' : 1.3,
    'tier2' : 1.1,
    'tier3' : 1,
    'rural' : 0.8
}
df['weighted_amount'] = df['city_tier'].map(region_mapping) * df['amount']
df.head()

In [ ]:
features = features.join(df[['log_loan_amount', 'weighted_amount']])
features.head()

## 贷款期限

In [ ]:
#计算月供和月供占家庭月收入比例
df['monthly_payment'] = round(df['amount'] / df['term'], 2)
df['repayment_ratio'] = np.where(
    df['family_income'] > 0,
    df['monthly_payment'] / df['family_income'],
    1
)
#业务分箱
ratio_bins = [0, 0.2, 0.3, np.inf]
ratio_labels = ['healthy', 'tight', 'warning']
df['repayment_ability'] = pd.cut(
    df['repayment_ratio'],
    bins=ratio_bins,
    labels=ratio_labels,
    right=False
)
df.head()

In [ ]:
features = features.join(df[['term', 'repayment_ability']])
features['repayment_ability'] = features['repayment_ability'].astype('category')
features.head()

In [ ]:
features.info()

## 居住状况

In [ ]:
housing_mapping = {
    1 : 'self_purchased',
    2 : 'mortaged',
    3 : 'parents_holding',
    4 : 'dorm',
    5 : 'rent',
    6 : 'shared',
    7 : 'other',
    8 : 'missing'
}
df['housing'] = df['housing'].map(housing_mapping)
df['housing'] = df['housing'].fillna('missing')
df['housing_city'] = (
            df['housing'].astype(str) + 
            '_in_' + 
            df['city_tier'].astype(str)
)
df.head()

In [ ]:
features = features.join(df['housing_city'])
features['housing_city'] = features['housing_city'].astype('category')
features.head()

## 担保方式

In [ ]:
#引入码值表，此处将担保方式合并为5大类
guarantee_mapping = {
    10 : 'promise',
    20 : 'mortage',
    40 : 'pledge',
    60 : 'bond'
}
df['guarantee'] = df['guarantee'].astype(str)
df['guarantee'] = df['guarantee'].fillna('missing')
guar_list = []
for method in df['guarantee']:
    if method == '5':
        guar_list.append('credit')
    elif int(method[:2]) in guarantee_mapping:
        guar_list.append(guarantee_mapping[int(method[:2])])
df['guarantee'] = guar_list
df.head()

In [ ]:
features = features.join(df['guarantee'])
features['guarantee'] = features['guarantee'].astype('category')
features.head()

## 额外的特征工程

In [ ]:
df.to_csv("train_features.csv", index=False)

In [ ]:
df["job_usage_interaction"] = df["job"].astype(str) + "_" + df["usage"].astype(str)

df["edu_usage_interaction"] = df["edu"].astype(str) + "_" + df["usage"].astype(str)

df["guarantee_housing_city_interaction"] = df["guarantee"].astype(str) + "_" + df["housing_city"].astype(str)

df["job_pos_city_interaction"] = (
    df["job"].astype(str) + "_" + df["pos"].astype(str) + "_" + df["housing_city"].astype(str)
)

df["job_edu_interaction"] = df["job"].astype(str) + "_" + df["edu"].astype(str)

df["job_housing_city_interaction"] = df["job"].astype(str) + "_" + df["housing_city"].astype(str)

df["repay_std_logloan_interaction"] = df["repay_std"].astype(str) + "_" + df["log_loan_amount"].round(1).astype(str)

In [ ]:
features = features.join(df[["job_usage_interaction", "edu_usage_interaction", "guarantee_housing_city_interaction",
                            "job_pos_city_interaction", "job_edu_interaction", "job_housing_city_interaction", "repay_std_logloan_interaction"]])

## EDA

In [ ]:
features.info()

In [ ]:
df.info()

In [ ]:
features = features.join(df['is_default'])
features.info()

In [ ]:
#first see age bin
grouped = df[['age_bin', 'is_default']].groupby('age_bin').mean().reset_index()

fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 8))
sns.countplot(x = 'age_bin', hue = 'is_default', data = features, ax = ax[0])
ax[0].set_yscale('log')
ax[0].set_xlabel('age_bin', fontsize = 12)
ax[0].set_ylabel('count', fontsize = 12)
ax[0].set_title('Default and Non-default in each Age Bin', fontsize = 14)
ax[0].legend()

sns.barplot(x = 'age_bin', y = 'is_default', data = df, ax = ax[1])
ax[1].set_xlabel('age_bin', fontsize = 12)
ax[1].set_ylabel('default_rate', fontsize = 12)
ax[1].set_title('Default Rate in Different Age Bin', fontsize = 14)
plt.show()

#### 从数据中看，老年人的违约率偏高，并且数据较少导致置信区间极大，可能需要未来对老年人贷款进行更多维度的更进一步的分析

In [ ]:
#first see age bin
grouped = df[['age', 'is_default']].groupby('age').mean().reset_index()
hist_kws={'histtype': 'bar', 'edgecolor':'black', 'alpha': 0.2}
fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 8))
sns.distplot(df[df['is_default'] == 0]['age'], label='Not Defaulted', 
             ax=ax[0], hist_kws=hist_kws)
sns.distplot(df[df['is_default'] == 1]['age'], label='Defaulted', 
             ax=ax[0], hist_kws=hist_kws)
ax[0].set_title('Count Plot of Age', fontsize=16)
ax[0].legend()
ax[1].plot(grouped['age'], grouped['is_default'], '.-')
ax[1].set_title('Default Rate vs. Age', fontsize=16)
ax[1].set_xlabel('age')
ax[1].set_ylabel('Default rate')
ax[1].grid(True)
plt.show()

#### 30-50岁中的违约概率明显比低龄与高龄贷款者要低

In [ ]:
#repay
print(len(df['repay_count'].unique()))
print(len(df['repay_std'].unique()))
print(len(df['repay_span'].unique()))
print(len(df['repay_ratio'].unique()))

In [ ]:
#repay_count
grouped = df[['repay_count', 'is_default']].groupby('repay_count').mean().reset_index()

hist_kws={'histtype': 'bar', 'edgecolor':'black', 'alpha': 0.2}
fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 8))
sns.distplot(df[df['is_default'] == 0]['repay_count'], label='Not Defaulted', 
             ax=ax[0], hist_kws=hist_kws)
sns.distplot(df[df['is_default'] == 1]['repay_count'], label='Defaulted', 
             ax=ax[0], hist_kws=hist_kws)
ax[0].set_title('Count Plot of Rapay Times', fontsize=16)
ax[0].legend()
ax[1].plot(grouped['repay_count'], grouped['is_default'], '.-')
ax[1].set_title('Default Rate vs. Repay Times', fontsize=16)
ax[1].set_xlabel('Repay Times')
ax[1].set_ylabel('Default rate')
ax[1].grid(True)
plt.show()

#### 可以看到高违约率集中在还款次数50次以下，所以如果当前流水还款次数小于50可能是高违约率的信号

In [ ]:
#repay_span
grouped = df[['repay_span', 'is_default']].groupby('repay_span').mean().reset_index()

hist_kws={'histtype': 'bar', 'edgecolor':'black', 'alpha': 0.2}
fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 8))
sns.distplot(df[df['is_default'] == 0]['repay_span'], label='Not Defaulted', 
             ax=ax[0], hist_kws=hist_kws)
sns.distplot(df[df['is_default'] == 1]['repay_span'], label='Defaulted', 
             ax=ax[0], hist_kws=hist_kws)
ax[0].set_title('Count Plot of Rapay Time Span', fontsize=16)
ax[0].legend()
ax[1].plot(grouped['repay_span'], grouped['is_default'], '.-')
ax[1].set_title('Default Rate vs. Repay Time Span', fontsize=16)
ax[1].set_xlabel('Repay Time Span')
ax[1].set_ylabel('Default rate')
ax[1].grid(True)
plt.show()

In [ ]:
#repay std
#点太稠密了，分组画个柱状图吧
df['repay_std_bin'] = pd.qcut(df['repay_std'], q = 5, duplicates = 'drop')
grouped = df.groupby('repay_std_bin')['is_default'].mean().reset_index()

plt.figure(figsize = (10,6))
sns.barplot(data = grouped, x = 'repay_std_bin', y = 'is_default')
plt.xticks(rotation = 45)
plt.ylabel('mean_default_rate', fontsize = 12)
plt.xlabel('repay_std_bin', fontsize = 12)
plt.title('Mean Default Rate under Different Repay std bin', fontsize = 14)
plt.show()

#### 可以清楚地看到当标准差上升的时候违约率也会变高

In [ ]:
#repay ratio
df['repay_ratio_bin'] = pd.qcut(df['repay_ratio'], q = 5, duplicates = 'drop')
grouped = df.groupby('repay_ratio_bin')['is_default'].mean().reset_index()

plt.figure(figsize = (10,6))
sns.barplot(data = grouped, x = 'repay_ratio_bin', y = 'is_default')
plt.xticks(rotation = 45)
plt.ylabel('mean_default_rate', fontsize = 12)
plt.xlabel('repay_ratio_bin', fontsize = 12)
plt.title('Mean Default Rate under Different Repay ratio bin', fontsize = 14)
plt.show()

#### 基本走势是还款占总额比例越高，违约率越低

In [ ]:
#marital status
grouped = df[['marital_status', 'is_default']].groupby('marital_status').mean().reset_index()

fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 8))
sns.countplot(x = 'marital_status', hue = 'is_default', data = features, ax = ax[0])
ax[0].set_yscale('log')
ax[0].set_xlabel('marital_status', fontsize = 12)
ax[0].set_ylabel('count', fontsize = 12)
ax[0].set_title('Default and Non-default in Different Marital Status', fontsize = 14)
ax[0].legend()

sns.barplot(x = 'marital_status', y = 'is_default', data = df, ax = ax[1])
ax[1].set_xlabel('marital_status', fontsize = 12)
ax[1].set_ylabel('default_rate', fontsize = 12)
ax[1].set_title('Default Rate in Different Marital Status', fontsize = 14)
plt.tight_layout()
plt.show()

#### 可以看到隐瞒婚姻状况者和离异、丧偶用户的违约风险偏高，且因为离异丧偶数据量偏少置信区间浮动大，所以同时也意味不确定性的提升

In [ ]:
#edu
grouped = df[['edu', 'is_default']].groupby('edu').mean().reset_index()

fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (20, 8))
sns.countplot(x = 'edu', hue = 'is_default', data = features, ax = ax[0])
ax[0].set_yscale('log')
ax[0].set_xlabel('edu', fontsize = 12)
ax[0].set_ylabel('count', fontsize = 12)
ax[0].set_title('Default and Non-default in Different EDU', fontsize = 14)
ax[0].legend()
plt.tight_layout()

sns.barplot(x = 'edu', y = 'is_default', data = df, ax = ax[1])
ax[1].set_xlabel('edu', fontsize = 12)
ax[1].set_ylabel('default_rate', fontsize = 12)
ax[1].set_title('Default Rate in Different EDU', fontsize = 14)
plt.show()

#### 可以看到，瞒报学历、中专、低学历（小学例外）违约率较高，大专虽然违约率看起来较低但是由于数据量少置信区间大，所以不确定性强

In [ ]:
#edu_city
grouped = df[['edu_city_interaction', 'is_default']].groupby('edu_city_interaction').mean().reset_index()
fig, ax = plt.subplots(nrows = 2, ncols = 1, figsize = (64, 32))
sns.barplot(df[df['is_default'] == 0]['edu_city_interaction'], label='Not Defaulted', 
             ax=ax[0], alpha = 0.2)
sns.barplot(df[df['is_default'] == 1]['edu_city_interaction'], label='Defaulted', 
             ax=ax[0], alpha = 0.2)
ax[0].set_title('Count Plot of Edu in Different City', fontsize=16)
ax[0].legend()
ax[1].plot(grouped['edu_city_interaction'], grouped['is_default'], '.-')
ax[1].set_title('Default Rate vs. Edu in Different City', fontsize=16)
ax[1].set_xlabel('Edu_city')
ax[1].set_ylabel('Default rate')
ax[1].grid(True)
plt.tight_layout()
plt.show()

#### 除了与之前单独edu匹配之外，总体违约率来看相同教育水平也是一线二线城市比较低一些

In [ ]:
#usage
grouped = df[['usage', 'is_default']].groupby('usage').mean().reset_index()
fig, ax = plt.subplots(nrows = 2, ncols = 1, figsize = (32, 16))
sns.barplot(df[df['is_default'] == 0]['usage'], label='Not Defaulted', 
             ax=ax[0], alpha = 0.2)
sns.barplot(df[df['is_default'] == 1]['usage'], label='Defaulted', 
             ax=ax[0], alpha = 0.2)
ax[0].set_title('Count Plot of Usage', fontsize=16)
ax[0].legend()
ax[1].plot(grouped['usage'], grouped['is_default'], '.-')
ax[1].set_title('Default Rate vs. Usage', fontsize=16)
ax[1].set_xlabel('Usage')
ax[1].set_ylabel('Default rate')
ax[1].grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#job
grouped = df[['job', 'is_default']].groupby('job').mean().reset_index()

fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (20, 8))
sns.countplot(x = 'job', hue = 'is_default', data = features, ax = ax[0])
ax[0].set_yscale('log')
ax[0].set_xlabel('job', fontsize = 12)
ax[0].set_ylabel('count', fontsize = 12)
ax[0].set_title('Default and Non-default in Different Job', fontsize = 14)
ax[0].legend()
plt.tight_layout()

sns.barplot(x = 'job', y = 'is_default', data = df, ax = ax[1])
ax[1].set_xlabel('job', fontsize = 12)
ax[1].set_ylabel('default_rate', fontsize = 12)
ax[1].set_title('Default Rate in Different Job', fontsize = 14)
plt.show()

In [ ]:
#income
plt.figure(figsize=(10,6))
sns.regplot(
    data = df,
    x = 'log_income',
    y = 'is_default',
    logistic = True, 
    scatter_kws = {'alpha': 0.2, 's': 10}
)
plt.xlabel('log_income')
plt.ylabel('is_default')
plt.title('Log income and Default')
plt.tight_layout()
plt.show()

#### 对数收入在8-10左右时有密集的高违约

In [ ]:
#income_missing
grouped = df[['income_missing', 'is_default']].groupby('income_missing').mean().reset_index()

fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 8))
sns.countplot(x = 'income_missing', hue = 'is_default', data = features, ax = ax[0])
ax[0].set_yscale('log')
ax[0].set_xlabel('income_missing', fontsize = 12)
ax[0].set_ylabel('count', fontsize = 12)
ax[0].set_title('Default and Non-default whether Income is Missing or not', fontsize = 14)
ax[0].legend()

sns.barplot(x = 'income_missing', y = 'is_default', data = df, ax = ax[1])
ax[1].set_xlabel('income_missing', fontsize = 12)
ax[1].set_ylabel('default_rate', fontsize = 12)
ax[1].set_title('Default Rate in Income Missing Condition', fontsize = 14)
plt.show()

#### 所以收入缺失确实是一个能够提供信息的标志 瞒报收入确实会有更高的违约风险

In [ ]:
#log family income
plt.figure(figsize=(10,6))
sns.regplot(
    data = df,
    x = 'log_family_income',
    y = 'is_default',
    logistic = True, 
    scatter_kws = {'alpha': 0.2, 's': 10}
)
plt.xlabel('log_family_income')
plt.ylabel('is_default')
plt.title('Log Family Income and Default')
plt.tight_layout()
plt.show()

#### 与income对应

In [ ]:
#log loan amount
df['log_loan_amount_bin'] = pd.qcut(df['log_loan_amount'], q = 5, duplicates = 'drop')
grouped = df.groupby('log_loan_amount_bin')['is_default'].mean().reset_index()

plt.figure(figsize = (10,6))
sns.barplot(data = grouped, x = 'log_loan_amount_bin', y = 'is_default')
plt.xticks(rotation = 45)
plt.ylabel('mean_default_rate', fontsize = 12)
plt.xlabel('log_loan_amount_bin', fontsize = 12)
plt.title('Mean Default Rate under Different log amount bin', fontsize = 14)
plt.show()

#### 大额贷款和中额贷款的违约率略高于小额

In [ ]:
#term
grouped = df[['term', 'is_default']].groupby('term').mean().reset_index()
hist_kws={'histtype': 'bar', 'edgecolor':'black', 'alpha': 0.2}
fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (16, 8))
sns.distplot(df[df['is_default'] == 0]['term'], label='Not Defaulted', 
             ax=ax[0], hist_kws=hist_kws)
sns.distplot(df[df['is_default'] == 1]['term'], label='Defaulted', 
             ax=ax[0], hist_kws=hist_kws)
ax[0].set_title('Count Plot of Term', fontsize=16)
ax[0].legend()
ax[1].plot(grouped['term'], grouped['is_default'], '.-')
ax[1].set_title('Default Rate vs. Term', fontsize=16)
ax[1].set_xlabel('Term')
ax[1].set_ylabel('Default rate')
ax[1].grid(True)
plt.show()

#### 长期贷款总体看来违约率更高一些

In [ ]:
#repayment_ability
grouped = df[['repayment_ability', 'is_default']].groupby('repayment_ability').mean().reset_index()

fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (20, 8))
sns.countplot(x = 'repayment_ability', hue = 'is_default', data = features, ax = ax[0])
ax[0].set_yscale('log')
ax[0].set_xlabel('repayment_ability', fontsize = 12)
ax[0].set_ylabel('count', fontsize = 12)
ax[0].set_title('Default and Non-default in Different Repayment Ability', fontsize = 14)
ax[0].legend()
plt.tight_layout()

sns.barplot(x = 'repayment_ability', y = 'is_default', data = df, ax = ax[1])
ax[1].set_xlabel('repayment_ability', fontsize = 12)
ax[1].set_ylabel('default_rate', fontsize = 12)
ax[1].set_title('Default Rate in Different Repayment Ability', fontsize = 14)
plt.show()

#### 健康的还款能力指标确实对应了更低的违约率

In [ ]:
#housing-city
grouped = df[['housing_city', 'is_default']].groupby('housing_city').mean().reset_index()

fig, ax = plt.subplots(nrows = 2, ncols = 1, figsize = (64, 32))
sns.countplot(x = 'housing_city', hue = 'is_default', data = features, ax = ax[0])
ax[0].set_yscale('log')
ax[0].set_xlabel('housing_city', fontsize = 12)
ax[0].set_ylabel('count', fontsize = 12)
ax[0].set_title('Default and Non-default in Different Housing Condition in Different City', fontsize = 14)
ax[0].legend()
plt.tight_layout()

sns.barplot(x = 'housing_city', y = 'is_default', data = df, ax = ax[1])
ax[1].set_xlabel('housing_city', fontsize = 12)
ax[1].set_ylabel('default_rate', fontsize = 12)
ax[1].set_title('Default Rate in Different Housing Condition in Different City', fontsize = 14)
plt.show()

In [ ]:
#guarantee
fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize = (20, 8))
sns.countplot(x = 'guarantee', hue = 'is_default', data = features, ax = ax[0])
ax[0].set_yscale('log')
ax[0].set_xlabel('guarantee', fontsize = 12)
ax[0].set_ylabel('count', fontsize = 12)
ax[0].set_title('Default and Non-default in Different Guarantees', fontsize = 14)
ax[0].legend()
plt.tight_layout()

sns.barplot(x = 'guarantee', y = 'is_default', data = df, ax = ax[1])
ax[1].set_xlabel('guarantee', fontsize = 12)
ax[1].set_ylabel('default_rate', fontsize = 12)
ax[1].set_title('Default Rate in Different Guarantees', fontsize = 14)
plt.show()

#### 质押明显的低违约率，按揭与保证的违约率较高，信用担保的数据量较少，置信区间极大，结果不稳定

## Machine Learning

In [ ]:
h2o.init()

In [ ]:
hf = H2OFrame(features)

In [ ]:
features.info()

In [ ]:
hf['age_bin'] = hf['age_bin'].asfactor()
hf['marital_status'] = hf['marital_status'].asfactor()
hf['edu'] = hf['edu'].asfactor()
hf['edu_city_interaction'] = hf['edu_city_interaction'].asfactor()
hf['usage'] = hf['usage'].asfactor()
hf['usage_city_interaction'] = hf['usage_city_interaction'].asfactor()
hf['job'] = hf['job'].asfactor()
hf['job_pos'] = hf['job_pos'].asfactor()
hf['income_missing'] = hf['income_missing'].asfactor()
hf['repayment_ability'] = hf['repayment_ability'].asfactor()
hf['housing_city'] = hf['housing_city'].asfactor()
hf['guarantee'] = hf['guarantee'].asfactor()
hf['is_default'] = hf['is_default'].asfactor()

In [ ]:
hf.summary()

In [ ]:
#split test and train data
strat_split = hf['is_default'].stratified_split(test_frac=0.25, seed=42)

train = hf[strat_split == 'train']
test = hf[strat_split == 'test']

feature = ['age_squared', 'age', 'age_bin', 'repay_ratio', 'repay_count', 'repay_std', 'repay_max', 'repay_span', 'marital_status',
          'edu', 'edu_city_interaction', 'usage', 'usage_city_interaction', 'job', 'job_city_interaction', 'job_pos',
          'log_income', 'income_missing', 'log_family_income', 'log_loan_amount', 'weighted_amount', 'term', 'repayment_ability',
          'housing_city', 'guarantee']
target = 'is_default'

In [ ]:
#实在太过不平衡，上采样一下
train_df = train.as_data_frame()
minority = train_df[train_df[target] == 1]
majority = train_df[train_df[target] == 0]

minority_upsampled = resample(minority,
                               replace=True,                  # 允许重复采样
                               n_samples=len(majority) // 2,  # 比多数类少一半
                               random_state=42)

majority_downsampled = resample(majority,
                                 replace=False,                 # 不重复采样
                                 n_samples=int(len(majority) * 0.8),  # 可根据需要调
                                 random_state=42)

balanced_df = pd.concat([minority_upsampled, majority_downsampled])

balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

train = h2o.H2OFrame(balanced_df)

In [ ]:
train['age_bin'] = train['age_bin'].asfactor()
train['marital_status'] = train['marital_status'].asfactor()
train['edu'] = train['edu'].asfactor()
train['edu_city_interaction'] = train['edu_city_interaction'].asfactor()
train['usage'] = train['usage'].asfactor()
train['usage_city_interaction'] = train['usage_city_interaction'].asfactor()
train['job'] = train['job'].asfactor()
train['job_pos'] = train['job_pos'].asfactor()
train['income_missing'] = train['income_missing'].asfactor()
train['repayment_ability'] = train['repayment_ability'].asfactor()
train['housing_city'] = train['housing_city'].asfactor()
train['guarantee'] = train['guarantee'].asfactor()
train['is_default'] = train['is_default'].asfactor()

In [ ]:
train['is_default'].table()

### DLC:测试新数据

In [ ]:
features = features.join(df['is_test'])

In [ ]:
train = features[features['is_test'].isnull()].drop('is_test', axis = 1)
test = features[features['is_test'] == 1].drop('is_test', axis = 1)
train.info()

In [ ]:
test.info()

In [ ]:
train = H2OFrame(train)
test = H2OFrame(test)

In [ ]:
feature = ['age_squared', 'age', 'age_bin', 'repay_ratio', 'repay_count', 'repay_std', 'repay_max', 'repay_span', 'marital_status',
          'edu', 'edu_city_interaction', 'usage', 'usage_city_interaction', 'job', 'job_city_interaction', 'job_pos',
          'log_income', 'income_missing', 'log_family_income', 'log_loan_amount', 'weighted_amount', 'term', 'repayment_ability',
          'housing_city', 'guarantee']
target = 'is_default'

In [ ]:
#实在太过不平衡，上采样一下
train_df = train.as_data_frame()
minority = train_df[train_df[target] == 1]
majority = train_df[train_df[target] == 0]

minority_upsampled = resample(minority,
                               replace=True,                  # 允许重复采样
                               n_samples=len(majority) // 2,  # 比多数类少一半
                               random_state=42)

majority_downsampled = resample(majority,
                                 replace=False,                 # 不重复采样
                                 n_samples=int(len(majority) * 0.8),  # 可根据需要调
                                 random_state=42)

balanced_df = pd.concat([minority_upsampled, majority_downsampled])

balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

train = h2o.H2OFrame(balanced_df)

In [ ]:
train['age_bin'] = train['age_bin'].asfactor()
train['marital_status'] = train['marital_status'].asfactor()
train['edu'] = train['edu'].asfactor()
train['edu_city_interaction'] = train['edu_city_interaction'].asfactor()
train['usage'] = train['usage'].asfactor()
train['usage_city_interaction'] = train['usage_city_interaction'].asfactor()
train['job'] = train['job'].asfactor()
train['job_pos'] = train['job_pos'].asfactor()
train['income_missing'] = train['income_missing'].asfactor()
train['repayment_ability'] = train['repayment_ability'].asfactor()
train['housing_city'] = train['housing_city'].asfactor()
train['guarantee'] = train['guarantee'].asfactor()
train['is_default'] = train['is_default'].asfactor()
test['age_bin'] = test['age_bin'].asfactor()
test['marital_status'] = test['marital_status'].asfactor()
test['edu'] = test['edu'].asfactor()
test['edu_city_interaction'] = test['edu_city_interaction'].asfactor()
test['usage'] = test['usage'].asfactor()
test['usage_city_interaction'] = test['usage_city_interaction'].asfactor()
test['job'] = test['job'].asfactor()
test['job_pos'] = test['job_pos'].asfactor()
test['income_missing'] = test['income_missing'].asfactor()
test['repayment_ability'] = test['repayment_ability'].asfactor()
test['housing_city'] = test['housing_city'].asfactor()
test['guarantee'] = test['guarantee'].asfactor()
test['is_default'] = test['is_default'].asfactor()

In [ ]:
train['weights'] = train['is_default'].ifelse(1.5, 1.0)
rf = H2ORandomForestEstimator(
    ntrees = 700,
    max_depth = 9,
    mtries = -1,
    min_rows = 1,
    sample_rate = 1,
    score_each_iteration = True,
    weights_column = 'weights',
    seed = 36
)
rf.train(x = feature, y = target, training_frame = train)

In [ ]:
# Make predictions and calculate AUC
train_true = train.as_data_frame()['is_default'].values
test_true = test.as_data_frame()['is_default'].values
train_pred = rf.predict(train).as_data_frame()['p1'].values
test_pred = rf.predict(test).as_data_frame()['p1'].values

train_fpr, train_tpr, _ = roc_curve(train_true, train_pred)
test_fpr, test_tpr, _ = roc_curve(test_true, test_pred)
train_auc = np.round(auc(train_fpr, train_tpr), 3)
test_auc = np.round(auc(test_fpr, test_tpr), 3)

In [ ]:
# 预测结果
preds = rf.predict(test)

# 查看结果
preds.head()

In [ ]:
preds_df = preds.as_data_frame()
test_df = test.as_data_frame()

# 合并看看具体客户违约风险
result = test_df.copy()
result["default_proba"] = preds_df["p1"]

# 查看前10个最可能违约的客户
result = result.sort_values(by="default_proba", ascending=False)
result.head()

In [ ]:
sns.kdeplot(result[result["is_default"] == 0]["default_proba"], label="non-default")
sns.kdeplot(result[result["is_default"] == 1]["default_proba"], label="default")
plt.title("Predicted Default Probability Distribution")
plt.xlabel("Probability of Default")
plt.legend()
plt.show()

In [ ]:
#找出预测违约率最高的前200人
top_k = result.sort_values(by='default_proba', ascending=False).head(200)
precision_at_k = top_k[(top_k['default_proba'] > 0.625)]['is_default'].mean()
recall_at_k = (top_k['is_default'] == 1).sum() / result['is_default'].sum()

print(f"Precision@200: {precision_at_k:.3f}")
print(f"Recall@200: {recall_at_k:.3f}")

### Random Forest

In [ ]:
train['weights'] = train['is_default'].ifelse(1.46, 1.0)
rf = H2ORandomForestEstimator(
    ntrees = 400,
    max_depth = 10,
    mtries = -1,
    min_rows = 1,
    sample_rate = 0.8,
    score_each_iteration = True,
    weights_column = 'weights',
    seed = 42
)
rf.train(x = feature, y = target, training_frame = train)

In [ ]:
# Feature importance
importance = rf.varimp(use_pandas = True)
fig, ax = plt.subplots(figsize = (10, 8))
sns.barplot(x = 'scaled_importance', y = 'variable', data = importance)
plt.show()

In [ ]:
# Make predictions and calculate AUC
train_true = train.as_data_frame()['is_default'].values
test_true = test.as_data_frame()['is_default'].values
train_pred = rf.predict(train).as_data_frame()['p1'].values
test_pred = rf.predict(test).as_data_frame()['p1'].values

train_fpr, train_tpr, _ = roc_curve(train_true, train_pred)
test_fpr, test_tpr, _ = roc_curve(test_true, test_pred)
train_auc = np.round(auc(train_fpr, train_tpr), 3)
test_auc = np.round(auc(test_fpr, test_tpr), 3)

In [ ]:
# Classification report
for thresh in [0.5, 0.4, 0.3, 0.2]:
    y_pred = (test_pred > thresh).astype(int)
    print(f"Threshold = {thresh}")
    print(classification_report(test_true, y_pred))

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(train_fpr, train_tpr, label = 'Train AUC: ' + str(train_auc))
ax.plot(test_fpr, test_tpr, label = 'Test AUC: ' + str(test_auc))
ax.set_xlabel('False Positive Rate', fontsize = 12)
ax.set_ylabel('True Positive Rate', fontsize = 12)
ax.legend(fontsize = 12)
plt.show()

In [ ]:
# 预测结果
preds = rf.predict(test)

# 查看结果
preds.head()

In [ ]:
perf = rf.model_performance(test)

print("AUC:", perf.auc())
print("LogLoss:", perf.logloss())
perf.plot()

In [ ]:
perf.find_threshold_by_max_metric("F1")

In [ ]:
preds_df = preds.as_data_frame()
test_df = test.as_data_frame()

# 合并看看具体客户违约风险
result = test_df.copy()
result["default_proba"] = preds_df["p1"]

# 查看前10个最可能违约的客户
result = result.sort_values(by="default_proba", ascending=False)
result.head()

In [ ]:
sns.kdeplot(result[result["is_default"] == 0]["default_proba"], label="non-default")
sns.kdeplot(result[result["is_default"] == 1]["default_proba"], label="default")
plt.title("Predicted Default Probability Distribution")
plt.xlabel("Probability of Default")
plt.legend()
plt.show()

In [ ]:
#找出预测违约率最高的前200人
top_k = result.sort_values(by='default_proba', ascending=False).head(200)
precision_at_k = top_k[(top_k['default_proba'] > 0.545)]['is_default'].mean()
recall_at_k = (top_k['is_default'] == 1).sum() / result['is_default'].sum()

print(f"Precision@200: {precision_at_k:.3f}")
print(f"Recall@200: {recall_at_k:.3f}")

### GBM

In [ ]:
hyper_params = {
    'max_depth': [2, 3, 4],
    'sample_rate': [0.7, 0.8, 1.0],
    'col_sample_rate': [0.7, 0.9, 1.0],
    'learn_rate': [0.01, 0.05, 0.1],
}

grid = H2OGridSearch(
    model = H2OGradientBoostingEstimator(
        ntrees = 100,
        stopping_rounds = 5,
        balance_classes = False,
        seed = 42
    ),
    hyper_params = hyper_params,
    grid_id = "h2o_gbm_grid"
)

grid.train(x = feature, y = target, training_frame = train)

# 按 AUC 取最优模型
sorted_grid = grid.get_grid(sort_by="auc", decreasing=True)
best_model = sorted_grid.models[0]
print("Best model AUC:", best_model.auc(valid=True))
best_model.varimp_plot()

In [ ]:
# 对训练集预测
train_pred = best_model.predict(train).as_data_frame()
y_train = train["is_default"].as_data_frame().values.flatten()
y_train_score = train_pred["p1"].values  # p1是违约概率

# 对测试集预测
test_pred = best_model.predict(test).as_data_frame()
y_test = test["is_default"].as_data_frame().values.flatten()
y_test_score = test_pred["p1"].values

# 计算 ROC 和 AUC
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_score)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_score)
auc_train = auc(fpr_train, tpr_train)
auc_test = auc(fpr_test, tpr_test)

# 画图
plt.figure(figsize=(8,6))
plt.plot(fpr_train, tpr_train, label=f"Train ROC (AUC = {auc_train:.3f})", color="blue")
plt.plot(fpr_test, tpr_test, label=f"Test ROC (AUC = {auc_test:.3f})", color="orange")
plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Train vs Test")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
perf = best_model.model_performance(test)

print("AUC:", perf.auc())
print("LogLoss:", perf.logloss())
perf.plot()

In [ ]:
preds_df = test_pred
test_df = test.as_data_frame()

# 合并看看具体客户违约风险
result = test_df.copy()
result["default_proba"] = preds_df["p1"]

# 查看前10个最可能违约的客户
result.sort_values(by="default_proba", ascending=False).head(10)

In [ ]:
sns.kdeplot(result[result["is_default"] == 0]["default_proba"], label="non-default")
sns.kdeplot(result[result["is_default"] == 1]["default_proba"], label="default")
plt.title("Predicted Default Probability Distribution")
plt.xlabel("Probability of Default")
plt.legend()
plt.show()

In [ ]:
perf.find_threshold_by_max_metric("F1")

In [ ]:
result['pred_default'] = (result['default_proba'] >= 0.1).astype(int)
confusion_table = result.sort_values(by = "default_proba", ascending = False)
confusion_table = confusion_table[['is_default', 'pred_default', 'default_proba']][0:200]
print(confusion_table)

In [ ]:
#找出预测违约率最高的前200人
top_k = result.sort_values(by='default_proba', ascending=False).head(200)
precision_at_k = (top_k['is_default'] == 1).mean()
recall_at_k = (top_k['is_default'] == 1).sum() / result['is_default'].sum()

print(f"Precision@200: {precision_at_k:.3f}")
print(f"Recall@200: {recall_at_k:.3f}")

### 简化模型防止过拟合

In [ ]:
print(features.columns)

In [ ]:
print(df.columns)

In [ ]:
x = features.copy()
x = x.drop(['edu_city_interaction', 'usage_city_interaction', 'job_city_interaction', 'job_usage_interaction',
           'edu_usage_interaction', 'guarantee_housing_city_interaction', 'job_pos_city_interaction',
           'job_edu_interaction', 'job_housing_city_interaction',
           'repay_std_logloan_interaction', 'housing_city', 'job_pos'], axis = 1)
x = x.join(df[['pos', 'city_tier', 'housing']])
hf = H2OFrame(x)

In [ ]:
hf.summary()

In [ ]:
hf['age_bin'] = hf['age_bin'].asfactor()
hf['marital_status'] = hf['marital_status'].asfactor()
hf['edu'] = hf['edu'].asfactor()
hf['usage'] = hf['usage'].asfactor()
hf['job'] = hf['job'].asfactor()
hf['pos'] = hf['pos'].asfactor()
hf['city_tier'] = hf['city_tier'].asfactor()
hf['housing'] = hf['housing'].asfactor()
hf['income_missing'] = hf['income_missing'].asfactor()
hf['repayment_ability'] = hf['repayment_ability'].asfactor()
hf['guarantee'] = hf['guarantee'].asfactor()
hf['is_default'] = hf['is_default'].asfactor()

In [ ]:
#split test and train data
strat_split = hf['is_default'].stratified_split(test_frac=0.25, seed=42)

train = hf[strat_split == 'train']
test = hf[strat_split == 'test']

feature = ['age_squared', 'age', 'age_bin', 'repay_ratio', 'repay_count', 'repay_std', 'repay_max', 'repay_span', 'marital_status',
          'edu', 'usage', 'job', 'city_tier', 'housing', 'pos',
          'log_income', 'income_missing', 'log_family_income', 'log_loan_amount', 'weighted_amount', 'term', 'repayment_ability',
          'guarantee']
target = 'is_default'

In [ ]:
train['is_default'].table()

In [ ]:
hyper_params = {
    'max_depth': [2, 3, 4],
    'sample_rate': [0.7, 0.8, 1.0],
    'col_sample_rate': [0.7, 0.9, 1.0],
    'learn_rate': [0.01, 0.05, 0.1],
}

grid = H2OGridSearch(
    model = H2OGradientBoostingEstimator(
        ntrees = 100,
        stopping_rounds = 5,
        balance_classes = True,
        seed = 42
    ),
    hyper_params = hyper_params,
    grid_id = "h2o_gbm_grid"
)

grid.train(x = feature, y = target, training_frame = train)

# 按 AUC 取最优模型
sorted_grid = grid.get_grid(sort_by="auc", decreasing=True)
best_model = sorted_grid.models[0]
print("Best model AUC:", best_model.auc(valid=True))
best_model.varimp_plot()

In [ ]:
# 对训练集预测
train_pred = best_model.predict(train).as_data_frame()
y_train = train["is_default"].as_data_frame().values.flatten()
y_train_score = train_pred["p1"].values  # p1是违约概率

# 对测试集预测
test_pred = best_model.predict(test).as_data_frame()
y_test = test["is_default"].as_data_frame().values.flatten()
y_test_score = test_pred["p1"].values

# 计算 ROC 和 AUC
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_score)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_score)
auc_train = auc(fpr_train, tpr_train)
auc_test = auc(fpr_test, tpr_test)

# 画图
plt.figure(figsize=(8,6))
plt.plot(fpr_train, tpr_train, label=f"Train ROC (AUC = {auc_train:.3f})", color="blue")
plt.plot(fpr_test, tpr_test, label=f"Test ROC (AUC = {auc_test:.3f})", color="orange")
plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Train vs Test")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
train_pd = train.as_data_frame()
test_pd = test.as_data_frame()

for col in x:
    if pd.api.types.is_numeric_dtype(train_pd[col]):
        plt.figure()
        sns.kdeplot(train_pd[col], label="train", linestyle="--")
        sns.kdeplot(test_pd[col], label="test")
        plt.title(col)
        plt.legend()
        plt.show()

### 气死了的时候运行这个

In [ ]:
h2o.cluster().shutdown()